# 12. Metadata Grouping과 Factorial Split 설계

이 노트북은 실제 `samples.csv`에서 평가 matrix와 학습/평가 manifest를 만듭니다.
이후 노트북은 여기서 저장한 manifest를 사용해 동일한 샘플 기준으로 비교합니다.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "ch2_utils.py").exists():
    matches = (
        list(Path.cwd().glob("Deeplearning/*/2-1장/ch2_utils.py"))
        + list(Path.cwd().glob("Deeplearning/*/2장/ch2_utils.py"))
        + list(Path.cwd().glob("**/ch2_utils.py"))
    )
    NOTEBOOK_DIR = matches[0].parent if matches else Path("Deeplearning") / "Vision 응용" / "2-1장"
sys.path.append(str(NOTEBOOK_DIR))

from ch2_utils import *

paths = find_paths()
set_korean_font()
set_seed(7)
samples = load_samples(paths.data_root)
paths

## 12-1. Factorial evaluation table 생성

In [ ]:
factorial_path = paths.runs_root / "factorial_eval_table.csv"
factorial = create_factorial_eval_table(samples, factorial_path)
display(factorial)
print(factorial_path)

## 12-2. Standard train/eval manifest 생성

In [ ]:
train_manifest, eval_manifest = create_standard_manifests(samples, paths.runs_root)
print("train_manifest:", train_manifest)
print("eval_manifest:", eval_manifest)
display(pd.read_csv(train_manifest).groupby(["color_group", "defect_type"]).size().reset_index(name="train_count"))
display(pd.read_csv(eval_manifest).groupby(["color_group", "defect_type"]).size().reset_index(name="eval_count"))

## 12-3. Red scratch 노출 비율 실험 manifest 생성

In [ ]:
exposure_summary = create_exposure_manifests(
    samples,
    ratios=[0.0, 0.1, 0.25, 0.5, 0.75],
    target_color="red",
    target_defect="scratch",
    train_size=360,
    eval_per_combo=12,
    seed=7,
    runs_root=paths.runs_root,
)
display(exposure_summary)

## 12-4. 평가 조합 heatmap

In [ ]:
pivot = factorial.pivot_table(index="color_group", columns="defect_type", values="n_samples", aggfunc="sum").fillna(0)
fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(pivot.values, cmap="Blues")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        ax.text(j, i, int(pivot.values[i, j]), ha="center", va="center")
ax.set_title("color x defect sample count")
fig.colorbar(im, ax=ax)
fig.tight_layout()
fig.savefig(paths.runs_root / "12_color_defect_counts.png", dpi=150)
plt.show()

## 12-5. 이 노트북의 결론

In [ ]:
missing = factorial[factorial["n_samples"] == 0]
print(
    f"결론: standard manifest와 exposure-ratio manifest를 저장했습니다. "
    f"실제 평가 가능한 color-shape-defect 조합은 {len(factorial)}개입니다."
)